In [ ]:
#installation des librairies
!pip install -q tensorflow==2.18.0 
!pip install -q segmentation-models==1.0.1

import os
os.environ["SM_FRAMEWORK"] = "tf.keras"

#import des librairies nécessaires à l'entrainement et au pré-traitement des données
import zipfile
import glob
import shutil
import json
import re
import random
from pathlib import Path


import numpy as np
import xml.etree.ElementTree as ET
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from segmentation_models import Unet
from segmentation_models.losses import bce_jaccard_loss
from segmentation_models.metrics import iou_score
from scipy import ndimage
import albumentations as A
import requests

In [ ]:
DATA_URL = "http://dag.cvc.uab.es/DATASETS/CVC-FP.zip"
DATA_DIR = Path("data")
ZIP_PATH = DATA_DIR / "dataset.zip"
EXTRACT_DIR = DATA_DIR / "CVC-FP" / "ImagesGT"

TRAIN_RATIO = 0.8

def download_data(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)

    if dest.exists():
        print(f"{dest} existe déjà, téléchargement annulé")
        return 0

    print(f"Téléchargement du dataset depuis {url}")
    response = requests.get(url, stream=True)
    response.raise_for_status()

    with open(dest, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

    print("Téléchargement fini")
    return 1

def unzip(zip_path, extract_to):
    print(f"Extraction {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_to)
    print("Extraction complète")
    os.remove(zip_path)

"""
Trouver les paires images/annotations dans le dataset à partir d'une liste.
Autorise aussi le formatage des conventions de noms.
En l'occurence, les annotations sont nommées {image}_gt_{nombre}, cette fonction permet d'ignorer ce qui se trouve après le "_gt_". 
"""
def collect_pairs_from_lists(images, labels, suffix="_gt_"):

    label_map = {}
    for label_path in labels:
        stem = label_path.stem
        if suffix in stem:
            prefix = stem.split(suffix)[0]
        else:
            prefix = stem
        label_map[prefix] = label_path

    pairs = []
    for img_path in images:
        img_stem = img_path.stem
        if img_stem in label_map:
            pairs.append((img_path, label_map[img_stem]))
        else:
            print(f"Pas d'annotation trouvée pour {img_stem}")

    return pairs

def organize_dataset(extract_dir, dest_dir, train_ratio=0.8):
    """
    Sépare les images et les labels et crée des dossiers respectifs
    """
    all_files = list(extract_dir.rglob("*"))
    images = [f for f in all_files if f.suffix.lower() == ".png"]
    labels = [f for f in all_files if f.suffix.lower() == ".svg"]

    pairs = collect_pairs_from_lists(images, labels, suffix="_gt_")
    print(f"{len(pairs)} paires image-label trouvées")

    pairs = list(pairs)
    random.shuffle(pairs)
    split_idx = int(len(pairs) * train_ratio)
    train_pairs = pairs[:split_idx]
    val_pairs = pairs[split_idx:]

    for split in ["train", "val"]:
        (dest_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (dest_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

    for img_path, lbl_path in train_pairs:
        shutil.copy(img_path, dest_dir / "images/train")
        shutil.copy(lbl_path, dest_dir / "labels/train")

    for img_path, lbl_path in val_pairs:
        shutil.copy(img_path, dest_dir / "images/val")
        shutil.copy(lbl_path, dest_dir / "labels/val")

def main():
    if download_data(DATA_URL, ZIP_PATH) : 
        unzip(ZIP_PATH, EXTRACT_DIR)
        organize_dataset(EXTRACT_DIR, DATA_DIR)
    shutil.rmtree('data/CVC-FP')

main()

In [ ]:

#redimension des images
def resize_with_padding(img, target=640):
    h, w = img.shape[:2]
    scale = target / max(h, w)
    nh, nw = int(h * scale), int(w * scale)

    img = cv2.resize(img, (nw, nh))
    pad_h = target - nh
    pad_w = target - nw

    top = pad_h // 2
    bottom = pad_h - top
    left = pad_w // 2
    right = pad_w - left

    img = cv2.copyMakeBorder(
        img, top, bottom, left, right,
        cv2.BORDER_CONSTANT, value=0
    )

    return img, scale, (top, left)


In [ ]:
#trouver les paires images/annotations dans le dataset à partir d'un dossier
def collect_pairs(img_dir, svg_dir):
    pairs = []

    svg_map = {}
    for svg_path in glob.glob(os.path.join(svg_dir, "*.svg")):
        name = Path(svg_path).stem
        if "_gt_" in name:
            prefix = name.split("_gt_")[0]
        else:
            prefix = name
        svg_map[prefix] = svg_path

    for img_path in glob.glob(os.path.join(img_dir, "*.png")):
        img_stem = Path(img_path).stem

        if img_stem in svg_map:
            pairs.append((img_path, svg_map[img_stem]))
        else:
            print("No SVG for:", img_stem)

    return pairs

train_pairs = collect_pairs(
    "data/images/train",
    "data/labels/train"
)

val_pairs = collect_pairs(
    "data/images/val",
    "data/labels/val"
)

In [ ]:
#définition des classes
CLASS_MAP = {
    "Other": 0,
    "Door": 1,
    "Window" : 2
}

#conversion des annotations svg vers des masques numpy
def svg_to_mask(svg_path, out_shape):
    tree = ET.parse(svg_path)
    root = tree.getroot()

    def find_tag(tag):
        return root.find(f".//{{*}}{tag}")

    width_el = find_tag("width")
    height_el = find_tag("height")

    if width_el is None or height_el is None:
        raise ValueError(f"SVG missing width/height: {svg_path}")

    svg_w = float(width_el.text)
    svg_h = float(height_el.text)

    H, W = out_shape

    scale_x = W / svg_w
    scale_y = H / svg_h

    mask = np.zeros((H, W), dtype=np.uint8)

    for el in root.iter():
        tag = el.tag.split("}")[-1]
        cls = el.attrib.get("class")
        if cls not in CLASS_MAP:
            continue

        cid = CLASS_MAP[cls]

        if tag == "polygon":
            pts = []
            for p in el.attrib["points"].split():
                x, y = map(float, p.split(","))
                pts.append([int(x * scale_x), int(y * scale_y)])

            pts = np.array(pts, np.int32)
            cv2.fillPoly(mask, [pts], cid)

    return mask

In [ ]:
#augmentation des données d'entrainement avec rotations verticales et horizontales
train_augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),

    A.Affine(
        translate_percent=0.02,
        scale=(0.95, 1.05),          
        rotate=(-2, 2),              
        interpolation=cv2.INTER_LINEAR,
        mask_interpolation=cv2.INTER_NEAREST,
        p=0.5
    ),
    A.GaussianBlur(blur_limit=3, p=0.2)
])

In [ ]:
#classe dans laquelle a lieu la boucle de pré-traitement des images et des masques
class FloorplanGenerator(tf.keras.utils.Sequence):
    def __init__(self, pairs, batch_size=4, augment=None, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.pairs = pairs
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.indices = np.arange(len(pairs))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.pairs) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx*self.batch_size:(idx+1)*self.batch_size]
    
        images = []
        masks = []
    
        for i in batch_idx:
            img_path, svg_path = self.pairs[i]
    
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            orig_h, orig_w = img.shape[:2]

            mask = svg_to_mask(svg_path, out_shape=(orig_h, orig_w))

            img, scale, (pad_y, pad_x) = resize_with_padding(img, target=640)

            mask = cv2.resize(
                mask,
                None,
                fx=scale,
                fy=scale,
                interpolation=cv2.INTER_NEAREST
            )

            mask = cv2.copyMakeBorder(
                mask,
                pad_y, 640 - mask.shape[0] - pad_y,
                pad_x, 640 - mask.shape[1] - pad_x,
                cv2.BORDER_CONSTANT,
                value=0
            )
            if self.augment:
                augmented = self.augment(image=img, mask=mask)
                img = augmented["image"]
                mask = augmented["mask"]
    
            images.append(img)
            masks.append(mask)
    
        images = np.array(images, dtype=np.float32) / 255.0
        masks = tf.one_hot(np.array(masks), depth=3)
    
        return images, masks


In [ ]:
#génération des données pré-traitées
train_gen = FloorplanGenerator(
    train_pairs,
    batch_size=4,
    augment=train_augment
)

val_gen = FloorplanGenerator(
    val_pairs,
    batch_size=4,
    augment=None,
    shuffle=False
)

In [ ]:
#classe de callback qui permet de plot les prédictions pendant l'entraiment pour évaluer l'apprentissage
class PredictionCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_gen, interval=1, num_samples=1):
        super().__init__()
        self.val_gen = val_gen
        self.interval = interval
        self.num_samples = num_samples

    def on_epoch_end(self, epoch, logs=None):
        if epoch % self.interval != 0:
            return
    
        plt.figure(figsize=(12, 4 * self.num_samples))
    
        # pick a random batch
        batch_idx = np.random.randint(len(self.val_gen))
        images, masks = self.val_gen[batch_idx]
        num_to_show = min(self.num_samples, images.shape[0])

        for i in range(num_to_show):
            img = images[i]
            gt_mask = tf.argmax(masks[i], axis=-1).numpy()
    
            # model prediction
            pred = self.model.predict(img[None, ...], verbose=0)[0]

            conf = np.max(pred, axis=-1)
            pred_mask = np.argmax(pred, axis=-1)
            pred_mask[conf < 0.3] = 0
    
            img_vis = (img * 255).astype(np.uint8)
    
            # overlays
            gt_overlay = img_vis.copy()
            gt_overlay[gt_mask == 1] = [255, 0, 0]
            gt_overlay[gt_mask == 2] = [0, 255, 0]
    
            pred_overlay = img_vis.copy()
            pred_overlay[pred_mask == 1] = [255, 0, 0]
            pred_overlay[pred_mask == 2] = [0, 255, 0]
    
            plt.subplot(self.num_samples, 3, 3*i + 1)
            plt.imshow(img_vis)
            plt.title("Image")
            plt.axis("off")
    
            plt.subplot(self.num_samples, 3, 3*i + 2)
            plt.imshow(gt_overlay)
            plt.title("Masque")
            plt.axis("off")
    
            plt.subplot(self.num_samples, 3, 3*i + 3)
            plt.imshow(pred_overlay)
            plt.title("Prediction")
            plt.axis("off")
            
    
        plt.tight_layout()
        plt.show()


In [ ]:
#métriques et loss utilisées ou testées

def iou_loss(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union        = tf.reduce_sum(y_true + y_pred, axis=[1,2,3]) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return 1 - tf.reduce_mean(iou)


def iou(y_true, y_pred, smooth=1e-6):
    y_true = y_true[..., 1:]  # drop background
    y_pred = y_pred[..., 1:]

    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true + y_pred) - intersection
    return (intersection + smooth) / (union + smooth)

def dice(y_true, y_pred, smooth=1e-6):
    y_true = y_true[..., 1:]
    y_pred = y_pred[..., 1:]

    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)
    return (2. * intersection + smooth) / (union + smooth)


def iou_loss(y_true, y_pred):
    return 1 - iou(y_true, y_pred)


def tversky_loss(alpha=0.7, beta=0.3, smooth=1e-6):
    def loss(y_true, y_pred):
        y_true_f = tf.reshape(y_true[..., 1:], [-1])
        y_pred_f = tf.reshape(y_pred[..., 1:], [-1])

        TP = tf.reduce_sum(y_true_f * y_pred_f)
        FP = tf.reduce_sum((1 - y_true_f) * y_pred_f)
        FN = tf.reduce_sum(y_true_f * (1 - y_pred_f))

        tversky = (TP + smooth) / (TP + alpha * FP + beta * FN + smooth)
        return 1.0 - tversky
    return loss

loss_fn = tversky_loss(alpha=0.7, beta=0.3)


ce = tf.keras.losses.CategoricalCrossentropy()

def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1, 3])
    y_pred_f = tf.reshape(y_pred, [-1, 3])
    intersection = tf.reduce_sum(y_true_f * y_pred_f, axis=0)
    denom = tf.reduce_sum(y_true_f + y_pred_f, axis=0)
    dice = (2. * intersection + smooth) / (denom + smooth)
    return 1 - tf.reduce_mean(dice)

def dice_ce_loss(y_true, y_pred):
    return 0.5 * ce(y_true, y_pred) + 0.5 * dice_loss(y_true, y_pred)



In [ ]:
#visualisation des images et des masques traités
#cette étape est cruciale pour s'assurer qu'il n'y ait pas de décalage entre la position des masques et des images
sample = train_gen[0]
img = sample[0][0]
mask = tf.argmax(sample[1][0], axis=-1)

plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.imshow(img); plt.title("Image d'entraînement")
plt.subplot(1,3,2); plt.imshow(mask == 1); plt.title("Masque des portes")
plt.subplot(1,3,3); plt.imshow(mask == 2); plt.title("Masque des fenêtres")
plt.show()


sample = val_gen[0]
img = sample[0][0]
mask = tf.argmax(sample[1][0], axis=-1)

plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.imshow(img); plt.title("Image de validation")
plt.subplot(1,3,2); plt.imshow(mask == 1); plt.title("Masque des portes")
plt.subplot(1,3,3); plt.imshow(mask == 2); plt.title("Masque des fenêtres")
plt.show()

num_classes=3

#création du modèle avec la dimension de nos images, le nombre de classe à prédire et la fonction d'activation
model = Unet('resnet34', 
             input_shape=(640,640,3), 
             classes=3, 
             activation='softmax'
)

#sauvegarde du modèle entrainer qui donne le meilleur résultat
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath="best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    mode='max'
)

#callback pour visualiser l'évolution lors du fitting
callback = PredictionCallback(
    val_gen=val_gen,
    interval=1,
    num_samples=2
)

#compilation du modèle avec l'optimizer AdamW (qui m'a donné de meilleur résultat que Adam) , la loss iou loss et la métrique iou
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=8e-5), 
    loss=dice_ce_loss,
    metrics=[
        dice,
        iou
    ]
)

#entrainement du modèle sur les données train pré-traitées
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    callbacks=[callback, checkpoint]
)

In [ ]:
def count_objects(mask):
    counts = {}
    for cls, name in [(1, "Door"), (2, "Window")]:
        labeled, num = ndimage.label(mask == cls)
        counts[name] = int(num)
    return counts

In [ ]:
#test du modèle, on pourra prendre des images d'autres datasets pour estimer la capacité de généralisation du modèle
val_folder = "data/images/val"
results = {}

for img_path in sorted(glob.glob(os.path.join(val_folder, "*.png"))):
    image = cv2.imread(img_path)
    image = cv2.resize(image, (640, 640))
    image = image / 255.0

    pred = model.predict(image[None, ...], verbose=0)[0]

    conf = np.max(pred, axis=-1)
    cls = np.argmax(pred, axis=-1)
    cls[conf < 0.4] = 0

    counts = count_objects(cls)

    img_name = os.path.basename(img_path)
    results[img_name] = counts

#on enregistre les prédictions sur le dataset de test dans un json pour avoir une trace écrite du nombre de portes et de fenêtres dans chaque image
with open("test_predictions.json", "w") as f:
    json.dump(results, f, indent=4)

print("Predictions sauvegardées pour toutes les images de test dans le fichier test_predictions.json")